#  Emotion Classification using Transformer Models

This notebook performs **emotion detection** on text data using several **Transformer-based models** — including BERT, RoBERTa, and DeBERTa variants.  
It covers the full machine learning pipeline: from **data exploration and cleaning** to **model training, evaluation, and comparison**.

---

##  Workflow Overview

### **Step 1 — Exploratory Data Analysis (EDA)**
- Load and inspect training and test datasets.  
- Check for missing values, duplicates, and empty strings.  
- Analyze label distributions and sentence length statistics.  
- Explore vocabulary size and detect encoding artifacts (e.g., “Â”).

### **Step 2 — Data Processing & Cleaning**
- Keep only relevant columns (`Sentence`, `Emotion`).  
- Drop duplicates and missing entries.  
- Fix text encoding issues and remove unwanted characters.  
- Normalize text:
  - Convert curly quotes, remove URLs and extra spaces.  
  - Lowercase all text.  
- Merge similar labels (e.g., `joy → happiness`).

### **Step 3 — Dataset Splitting**
- Split dataset into:
  - **Train (80%)**
  - **Validation (20%)**
  - **Independent Test Set**
- Stratified split maintains balanced label proportions.

---

### **Model Training and Evaluation**

After data preparation, we trained multiple **Transformer architectures** for emotion classification:

- **BERT-base** as the initial baseline model.  
- **RoBERTa-base** and **RoBERTa-large** for improved contextual understanding.  
- **Weighted RoBERTa-large** to handle class imbalance.  
- **DeBERTa-v3-large** (with and without data augmentation) for state-of-the-art performance.

Each model was **fine-tuned** on the training set and evaluated using:
- **Accuracy**
- **Macro and Weighted F1-scores**
- **Confusion matrices** to analyze per-class performance

Finally, we compared results across all models to identify the best-performing configuration for emotion detection.

---


# Step 1 – Exploratory Data Analysis (EDA)

### 1.1 Imports + load data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd
from collections import Counter
import random
import numpy as np
import torch
from transformers import set_seed
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

# Ensure deterministic operations (slower but stable)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seeds fixed. Results will now be reproducible.")


# # # Loading training file
train_df = pd.read_excel("../data/nnnhis_year_and_last_year_cleanedmix_shuffled1.xlsx")   

print("Training data shape:", train_df.shape)
print(train_df.head())

# # # Loading test file

test_df = pd.read_excel("../data/group_23_url_1_transcript.xlsx")    

print("Test data shape:", test_df.shape)
print(test_df.head())

### 1.2 :Missing values and duplicates &  Empty strings check

In [ ]:
print("Missing values in train:\n", train_df.isna().sum())
print("\nMissing values in test:\n", test_df.isna().sum())

print("\nDuplicate rows in train:", train_df.duplicated().sum())
print("Duplicate rows in test:", test_df.duplicated().sum())

print("Empty strings in train:", (train_df["sentence"].astype(str).str.strip() == "").sum())
print("Empty strings in test:", (test_df["Sentence"].astype(str).str.strip() == "").sum())

### 1.3: Labels distribution

In [ ]:
print("Train labels:\n", train_df["emotion"].value_counts())
print("\nTest labels:\n", test_df["Emotion_core"].value_counts())

train_df["emotion"].value_counts().plot(kind="bar", title="Train emotions", figsize=(6,4))
plt.show()

test_df["Emotion_core"].value_counts().plot(kind="bar", title="Test emotions", figsize=(6,4))
plt.show()

### 1.4: Sentence length stats

In [ ]:
# Add sentence length column to training set
train_df["len_words"] = train_df["sentence"].astype(str).apply(lambda x: len(x.split()))

# Add sentence length column to test set
test_df["len_words"] = test_df["Sentence"].astype(str).apply(lambda x: len(x.split()))
train_df["len_words"].hist(bins=30, figsize=(6,4))
plt.title("Train sentence length")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.show()

test_df["len_words"].hist(bins=30, figsize=(6,4))
plt.title("Test sentence length")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.show()


### 1.5:Vocabulary size and top words

In [ ]:
train_text = " ".join(train_df["sentence"].astype(str)).lower()
train_text = re.sub(r"[^\w\s']", " ", train_text)
train_tokens = train_text.split()

test_text = " ".join(test_df["Sentence"].astype(str)).lower()
test_text = re.sub(r"[^\w\s']", " ", test_text)
test_tokens = test_text.split()

print("Train vocab size:", len(set(train_tokens)))
print("Test vocab size:", len(set(test_tokens)))

print("\nTop 20 train words:", Counter(train_tokens).most_common(20))

print("Train 'Â' count:", train_df["sentence"].astype(str).str.contains("Â").sum())
print("Test 'Â' count:", test_df["Sentence"].astype(str).str.contains("Â").sum())



---------------------------
# Step 2 – Data Processing (Cleaning)

### 2.1  Keep only the columns we need and rename them


In [ ]:
train_df = train_df[["sentence", "emotion"]]
train_df = train_df.rename(columns={"sentence": "Sentence", "emotion": "Emotion"})

test_df = test_df[["Sentence", "Emotion_core"]]
test_df = test_df.rename(columns={"Emotion_core": "Emotion"})
print(" Train shape:", train_df.shape)
print(" Test shape:", test_df.shape)
print(train_df.head(), "\n")
print(test_df.head(), "\n")

### 2.2 Drop missing values and duplicates


In [ ]:
train_df = train_df.dropna(subset=["Sentence", "Emotion"])
test_df = test_df.dropna(subset=["Sentence", "Emotion"])

train_df = train_df.drop_duplicates(subset=["Sentence", "Emotion"])
test_df = test_df.drop_duplicates(subset=["Sentence", "Emotion"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape, "\n")
print("\nDuplicate rows in train:", train_df.duplicated().sum())
print("Duplicate rows in test:", test_df.duplicated().sum())

### 2.3: Fixing  label mismatch: joy to happiness


In [ ]:
train_df["Emotion"] = train_df["Emotion"].replace({"joy": "happiness"})

print("Train labels:\n", train_df["Emotion"].value_counts(), "\n")


### 2.4: Remove weird characters like Â &  Lowercase

In [ ]:
# Remove weird characters like Â

train_df["Sentence"] = train_df["Sentence"].astype(str).str.replace("Â", "", regex=False)
test_df["Sentence"]  = test_df["Sentence"].astype(str).str.replace("Â", "", regex=False)

print("Example train:", train_df["Sentence"].iloc[0])
print("Example test:", test_df["Sentence"].iloc[0], "\n")


In [ ]:

# Text Cleaning: Fix Encoding Artifacts Instead of Dropping


import re
import pandas as pd

def fix_text_encoding(text):
    """
    Fix common UTF-8 encoding artifacts (â€™, â€œ, etc.)
    and normalize spacing and punctuation.
    """
    if not isinstance(text, str):
        return text

    replacements = {
        "â€™": "'",   # apostrophe
        "â€˜": "'",   # left single quote
        "â€œ": '"',   # left double quote
        "â€": '"',   # right double quote
        "â€“": "-",   # en dash
        "â€”": "-",   # em dash
        "â€¦": "...", # ellipsis
        "Â": "",      # stray "Â" before symbols
        "â€": "",     # remaining fragments
        "Ã©": "é",    # accented e
        "Ã": "A",     # capital A
        "�": "",      # unknown symbol
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # remove multiple spaces, trim whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


#  Apply to your data columns (e.g., train_df, val_df, test_df)
for df in [train_df]:
    df["Sentence"] = df["Sentence"].astype(str).apply(fix_text_encoding)

print(" Encoding artifacts fixed — sentences preserved!")


### 2.5: Remove URLs,keep only letters, apostrophes and spaces and remove extra spaces

In [ ]:
# Normalize curly apostrophes ’ → '
train_df["Sentence"] = train_df["Sentence"].str.replace("’", "'", regex=False)
test_df["Sentence"]  = test_df["Sentence"].str.replace("’", "'", regex=False)

# Remove URLs
train_df["Sentence"] = train_df["Sentence"].apply(lambda x: re.sub(r"http\S+|www\S+", " ", x))
test_df["Sentence"]  = test_df["Sentence"].apply(lambda x: re.sub(r"http\S+|www\S+", " ", x))

print("Example train:", train_df["Sentence"].iloc[0])
print("Example test:", test_df["Sentence"].iloc[0], "\n")

# Remove extra spaces
train_df["Sentence"] = train_df["Sentence"].str.replace(r"\s+", " ", regex=True).str.strip()
test_df["Sentence"]  = test_df["Sentence"].str.replace(r"\s+", " ", regex=True).str.strip()

print("Example train:", train_df["Sentence"].iloc[0])
print("Example test:", test_df["Sentence"].iloc[0], "\n")


### 3.1:  Split the Data

In [ ]:
from sklearn.model_selection import train_test_split

# Split training set into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    train_df["Sentence"], train_df["Emotion"],
    test_size=0.2, random_state=42, stratify=train_df["Emotion"]
)

# Test set stays the same
X_test = test_df["Sentence"]
y_test = test_df["Emotion"]

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))


### We split our dataset into train, validation, and test sets:

- **Training set (41,849 sentences)** → used to fit the model and learn patterns.  
- **Validation set (10,463 sentences)** → used to tune parameters and check performance before touching the test set.  
- **Test set (638 sentences)** → kept completely separate to fairly evaluate the final model.  

This way, the model doesn’t “cheat” by seeing the test data during training, and we can trust the test results as a real measure of performance.

-------

# 4: Modeling 

### 4.1 Imports + Setup

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


-----------------
### 4.2: Label Encoding

Before training, we need to convert the emotion labels (strings) into numerical IDs, since models work with numbers, not text.  

- `label_list`: extracts all unique emotion labels from the training data and sorts them to ensure a consistent order.  
- `label2id`: dictionary mapping each emotion (e.g., "happy", "sad") to a unique integer ID.  
- `id2label`: reverse mapping, used later to convert predictions back into human-readable labels.  

This step ensures that our model knows how to interpret labels during training and evaluation.


In [ ]:
# Create label → id mapping
label_list = sorted(train_df["Emotion"].unique().tolist())  
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print("Label mapping:", label2id)


----------
### 4.3: Converting DataFrames to Hugging Face Datasets

To train a transformer model, we need our data in the datasets.Dataset format used by Hugging Face.  

- We first create three datasets (train_dataset, val_dataset, test_dataset) from pandas DataFrames.  
- Each dataset has two columns:  
  - `"text"` → the input sentences.  
  - `"label"` → the encoded label IDs (using label2id).  
- Finally, we combine them into a single `DatasetDict with three splits: **train**, **validation**, and **test**.  

This format is compatible with the Hugging Face `Trainer` API and makes preprocessing and training much easier.


In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(pd.DataFrame({
    "text": X_train,
    "label": y_train.map(label2id)   # convert text → id
}))
val_dataset = Dataset.from_pandas(pd.DataFrame({
    "text": X_val,
    "label": y_val.map(label2id)
}))
test_dataset = Dataset.from_pandas(pd.DataFrame({
    "text": X_test,
    "label": y_test.map(label2id)
}))

# Combine into DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(dataset)


### 4.4 Tokenization with BERT

In [ ]:
# 4.4: Tokenization with BERT 
from transformers import AutoTokenizer, DataCollatorWithPadding

# Load pretrained BERT tokenizer
bert_checkpoint = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_checkpoint)

#  Define tokenization function (uses BERT tokenizer)
def tokenize_function_bert(example):
    return bert_tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

#  Tokenize all splits (train / validation / test)
tokenized_datasets_bert = dataset.map(
    tokenize_function_bert,
    batched=True,
    remove_columns=["text"]
)

#  Remove leftover index columns if they exist
for split in tokenized_datasets_bert.keys():
    if "__index_level_0__" in tokenized_datasets_bert[split].column_names:
        tokenized_datasets_bert[split] = tokenized_datasets_bert[split].remove_columns(["__index_level_0__"])

# Collator for dynamic padding
bert_data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

print(" Columns after cleanup:", tokenized_datasets_bert["train"].column_names)
print(" BERT Tokenization complete.")

### 4.5 : Load BERT Model & Training Configuration

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from transformers import EarlyStoppingCallback

# Load pretrained BERT model
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
print(" Loaded model: BERT-base-uncased")

#  Define training arguments
bert_training_args = TrainingArguments(
    output_dir="bert_emotion_results",    
    eval_strategy="epoch",           
    save_strategy="epoch",                
    learning_rate=2e-5,                    
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
    seed=42
)

# Define evaluation metrics
def compute_metrics_bert(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# Initialize Trainer
bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=tokenized_datasets_bert["train"],
    eval_dataset=tokenized_datasets_bert["validation"],
    tokenizer=bert_tokenizer,
    data_collator=bert_data_collator,
    compute_metrics=compute_metrics_bert,
    
)

print("Trainer initialized — starting BERT fine-tuning...")


### 4.6 : Train Bert


In [ ]:
bert_trainer.train()

### 4.7 : Evaluate BERT-base on Test Set

In [ ]:
predictions = bert_trainer.predict(tokenized_datasets_bert["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Classification metrics
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

# ---- Save metrics into variables ----
bert_acc = round(accuracy_score(y_true, y_pred), 3)
bert_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
bert_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print for confirmation
print(f"Macro F1: {bert_f1_macro}")
print(f"Weighted F1: {bert_f1_weighted}")
print(f"Accuracy: {bert_acc}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_list, yticklabels=label_list)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — BERT-base")
plt.tight_layout()
plt.show()

print(" Evaluation complete.")


--------------------------------

# 5 : RoBERTa-base 




### 5.1: Tokenization with RoBERTa

We use the **RoBERTa tokenizer** (`roberta-base`) to turn sentences into numbers (tokens) that the model can understand.  
- **Truncation**: if a sentence is too long, it gets cut to fit the limit (512 tokens).  
- **Data collator with padding**: this makes sure that all sentences in a batch have the same length by adding padding only where needed.  

In [ ]:

# 4.7: Tokenization + Fine-Tuning with RoBERTa-base 

from transformers import AutoTokenizer, DataCollatorWithPadding

# Load pretrained RoBERTa tokenizer
roberta_checkpoint = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_checkpoint)

#  Tokenization function
def tokenize_function_roberta(example):
    return roberta_tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

# Apply tokenization to all dataset splits
tokenized_datasets_roberta = dataset.map(
    tokenize_function_roberta,
    batched=True,
    remove_columns=["text"]
)

#  Remove leftover index columns if they exist
for split in tokenized_datasets_roberta.keys():
    if "__index_level_0__" in tokenized_datasets_roberta[split].column_names:
        tokenized_datasets_roberta[split] = tokenized_datasets_roberta[split].remove_columns(["__index_level_0__"])

#  Create collator for dynamic padding
roberta_data_collator = DataCollatorWithPadding(tokenizer=roberta_tokenizer)

print(" Tokenization complete — columns:", tokenized_datasets_roberta["train"].column_names)


### 5.2 Loading the RoBERTa Model for Classification  

We load a **pretrained RoBERTa model** and adapt it for our emotion classification task.  

- `num_labels` tells the model how many different emotions we have.  
- `id2label` and `label2id` link the numeric IDs back to the original emotion names (so results are human-readable).  
- The model is now ready to learn from our dataset and predict emotion categories for new text.  


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Load pretrained RoBERTa-base
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
print("Loaded model: RoBERTa-base")

### 5.3 : Setting Training Arguments  

Here we set the rules for training the RoBERTa model. We tell it where to save results, how often to evaluate and save (after every epoch), and use a small learning rate (2e-5) for stable learning. The model trains in batches of 16 samples for 3 full rounds (epochs) with a bit of weight decay (0.01) to avoid overfitting. We also log progress, and at the end, keep the version of the model that gives the best F1 score.  


In [ ]:
# Define training arguments
roberta_training_args = TrainingArguments(
    output_dir="roberta_emotion_results",
    eval_strategy="epoch",       
    save_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
    seed=4
)

# Metrics function
def compute_metrics_roberta(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# Initialize Trainer
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=tokenized_datasets_roberta["train"],
    eval_dataset=tokenized_datasets_roberta["validation"],
    tokenizer=roberta_tokenizer,
    data_collator=roberta_data_collator,
    compute_metrics=compute_metrics_roberta,
   

)

### 5.5: Training the Model

We start training the RoBERTa model using the Trainer. It will run for the number of epochs we set in TrainingArguments and save the best model.

In [ ]:
print(" Trainer initialized — starting RoBERTa fine-tuning...")
roberta_trainer.train()


###  5.6 : Evaluate the Baseline RoBERTa Model on the Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#  Run predictions on test set
predictions = roberta_trainer.predict(tokenized_datasets_roberta["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Detailed classification report
print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

#  Summary metrics 
roberta_acc = round(accuracy_score(y_true, y_pred), 3)
roberta_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
roberta_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print summary
print(f"\n Test Results Summary:")
print(f"Accuracy:     {roberta_acc:.3f}")
print(f"Macro F1:     {roberta_f1_macro:.3f}")
print(f"Weighted F1:  {roberta_f1_weighted:.3f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_list,
    yticklabels=label_list
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — RoBERTa-base")
plt.tight_layout()
plt.show()

print(" Evaluation complete.")


-------------------------------------------
# 6 : RoBERTa with Sentements


### 6.1 : Imports and Initialization for Sentiment Analysis with VADER, Transformers, and PyTorch

In [ ]:
# 5.1: Imports and Initialization for RoBERTa with Sentiment Features
import numpy as np
import pandas as pd
import torch
from torch import nn
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download and initialize VADER sentiment analyzer
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()


### 6.2 : Function to Add VADER Sentiment Features to Text Data

In [ ]:
# 5.2: Compute VADER Sentiment Scores for Train and Validation Data
# This step adds four sentiment features (neg, neu, pos, compound)
# using NLTK's VADER sentiment analyzer. 
# We only compute them for training and validation data 
# to avoid feature leakage into the test set.

def add_sentiment_features(df, text_col="Sentence"):
    """Apply VADER sentiment analysis to each text and return a DataFrame with sentiment scores."""
    scores = df[text_col].astype(str).apply(lambda x: sia.polarity_scores(x))
    sentiment_df = pd.DataFrame(scores.tolist(), index=df.index)[
        ["neg", "neu", "pos", "compound"]
    ]
    df = pd.concat([df, sentiment_df], axis=1)
    return df

#  Apply sentiment features to train and validation sets only
train_sent_df = add_sentiment_features(pd.DataFrame({"Sentence": X_train}))
val_sent_df = add_sentiment_features(pd.DataFrame({"Sentence": X_val}))

#  Test set will not include sentiment features (to prevent data leakage)
test_sent_df = pd.DataFrame({"Sentence": X_test})

# Quick check
train_sent_df.head()


### 6.3 : Creating Hugging Face Datasets with Sentiment Features and Label Mapping

In [ ]:
# 5.3: Create Hugging Face Datasets with Sentiment Features
# We now convert our train, validation, and test splits into
# Hugging Face Datasets. The model will receive both text and
# VADER sentiment features (neg, neu, pos, compound) during training
# — but NOT for the test set, to prevent data leakage.

from datasets import Dataset, DatasetDict

# Map emotion labels to numeric IDs
label_list = sorted(y_train.unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

def make_hf_dataset_with_sentiment(texts, labels, sentiment_df):
    """Build a Hugging Face Dataset containing text, labels, and VADER sentiment features."""
    df = pd.DataFrame({
        "text": texts,
        "label": labels.map(label2id),
        "neg": sentiment_df["neg"],
        "neu": sentiment_df["neu"],
        "pos": sentiment_df["pos"],
        "compound": sentiment_df["compound"],
    })
    return Dataset.from_pandas(df)

#  Train and validation include sentiment features
train_dataset_sent = make_hf_dataset_with_sentiment(X_train, y_train, train_sent_df)
val_dataset_sent   = make_hf_dataset_with_sentiment(X_val, y_val, val_sent_df)

#  Test set excludes sentiment columns (no data leakage)
test_dataset_sent = Dataset.from_pandas(pd.DataFrame({
    "text": X_test,
    "label": y_test.map(label2id)
}))

# Combine into one DatasetDict
dataset_sent = DatasetDict({
    "train": train_dataset_sent,
    "validation": val_dataset_sent,
    "test": test_dataset_sent,
})


### 6.4 : Tokenizing Text and Integrating Sentiment Features for Transformer Input

In [ ]:
# 5.4: Tokenization and Sentiment Integration (RoBERTa-base)
# This step tokenizes our text data for RoBERTa-base.
# For training and validation, we include VADER sentiment scores (neg, neu, pos, compound).
# The test split is handled safely — it does not require sentiment columns.

from transformers import AutoTokenizer, DataCollatorWithPadding

# Load the RoBERTa-base tokenizer
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_with_sentiment(batch):
    """Tokenize text and attach sentiment features if available."""
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    #  Add sentiment features only if present (train/val)
    if "neg" in batch:
        tokens["neg"] = batch["neg"]
        tokens["neu"] = batch["neu"]
        tokens["pos"] = batch["pos"]
        tokens["compound"] = batch["compound"]

    #  Always include labels
    tokens["labels"] = batch["label"]
    return tokens

# Apply tokenization to all splits
tokenized_datasets_sent = dataset_sent.map(
    tokenize_with_sentiment,
    batched=True,
    remove_columns=["text"],
)

# Remove leftover index columns
for split in tokenized_datasets_sent.keys():
    if "__index_level_0__" in tokenized_datasets_sent[split].column_names:
        tokenized_datasets_sent[split] = tokenized_datasets_sent[split].remove_columns(["__index_level_0__"])

# Sanity check: ensure all expected columns exist
for split in tokenized_datasets_sent.keys():
    print(f"{split} columns → {tokenized_datasets_sent[split].column_names}")

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


### 6.5 : Custom RoBERTa Model Enhanced with VADER Sentiment Features

In [ ]:
# 5.5: Custom RoBERTa Model Enhanced with VADER Sentiment Features
# This model combines RoBERTa's [CLS] embedding with four additional
# sentiment features (neg, neu, pos, compound) from VADER.
# It safely handles missing sentiment inputs ( test set).

import torch
import torch.nn as nn
from transformers import AutoModel

class RobertaWithSentiment(nn.Module):
    def __init__(self, checkpoint, num_labels):
        super().__init__()
        # Load pretrained RoBERTa encoder
        self.roberta = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.roberta.config.hidden_size

        # Classifier takes [CLS] embedding + 4 sentiment features
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 4, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels),
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        neg=None,
        neu=None,
        pos=None,
        compound=None,
        labels=None,
    ):
        # Encode input text with RoBERTa
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token

        #  Safely handle missing sentiment features (e.g., test split)
        if neg is not None and neu is not None and pos is not None and compound is not None:
            sentiment_features = torch.stack([neg, neu, pos, compound], dim=1)
        else:
            sentiment_features = torch.zeros(
                (pooled_output.size(0), 4),
                device=pooled_output.device,
                dtype=pooled_output.dtype,
            )

        # Concatenate [CLS] + sentiment features
        x = torch.cat([pooled_output, sentiment_features], dim=1)
        logits = self.classifier(x)

        # Compute loss if labels are provided
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


### 6.6 : Training Configuration and Evaluation Metrics for RoBERTa with Sentiment Features

In [ ]:
# 5.6: Training Configuration and Evaluation Metrics
# This section defines the training setup and evaluation metrics
# for the RoBERTa-base model enhanced with VADER sentiment features.

from transformers import TrainingArguments

#  Training hyperparameters and configuration
training_args = TrainingArguments(
    output_dir="roberta_sent_results",    
    eval_strategy="epoch",           
    save_strategy="epoch",                
    learning_rate=1.5e-5,                    
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
    seed=42
)

#  Metrics function for evaluation
def compute_metrics(eval_pred):
    """Compute accuracy and F1 scores for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


### 6.7 : Initializing Trainer with Custom RoBERTa-Sentiment Model

In [ ]:
# 5.7: Initialize Trainer with Custom RoBERTa + Sentiment Model
# We now combine our model, tokenized datasets, training arguments, 
# and evaluation metrics into a Hugging Face Trainer.

#  Initialize the custom model
model = RobertaWithSentiment(
    checkpoint=checkpoint,
    num_labels=len(label_list)
)

#  Build the Trainer (handles training, evaluation, logging, and checkpointing)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets_sent["train"],
    eval_dataset=tokenized_datasets_sent["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)


### 6.8 : Train the Model

In [ ]:
trainer.train()

### 6.9 : Evaluate on Test Set¶

In [ ]:
#  Evaluate the RoBERTa + Sentiment Model on the Test Set
# Evaluates model performance using accuracy, F1 scores, and confusion matrix.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

# Run predictions on the test split
predictions = trainer.predict(tokenized_datasets_sent["test"])

# Extract predicted and true labels
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Detailed classification report
print(" Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

#  Save metrics for summary table
roberta_sent_acc = round(accuracy_score(y_true, y_pred), 3)
roberta_sent_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
roberta_sent_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print summary
print(f"\n Test Results Summary:")
print(f"Accuracy:     {roberta_sent_acc:.3f}")
print(f"Macro F1:     {roberta_sent_f1_macro:.3f}")
print(f"Weighted F1:  {roberta_sent_f1_weighted:.3f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_list,
    yticklabels=label_list
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — RoBERTa + Sentiment Model")
plt.tight_layout()
plt.show()
print(" Evaluation complete.")

# 7 : RoBERTa-Large

In [ ]:
# RoBERTa-Large — Baseline 

# This version uses the RoBERTa-Large model for emotion classification
# on the text only .

from transformers import AutoTokenizer, AutoModelForSequenceClassification

#  Load pretrained RoBERTa-Large
checkpoint = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

#  Initialize model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_list),   
    id2label=id2label,
    label2id=label2id,
)

print("\n RoBERTa-Large model initialized successfully.")

### 7.1 : Tokenization for RoBERTa-Large

In [ ]:
# Clean all splits to remove sentiment columns for pure RoBERTa-large baseline
for split in dataset_sent.keys():
    cols_to_remove = [c for c in ["neg", "neu", "pos", "compound"] 
                      if c in dataset_sent[split].column_names]
    if cols_to_remove:
        dataset_sent[split] = dataset_sent[split].remove_columns(cols_to_remove)

#  Tokenization for RoBERTa-large 
def tokenize_function(batch):
    """Tokenize text data for RoBERTa-Large baseline."""
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

tokenized_datasets_nosent = dataset_sent.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

#  Clean index column if needed
for split in tokenized_datasets_nosent.keys():
    if "__index_level_0__" in tokenized_datasets_nosent[split].column_names:
        tokenized_datasets_nosent[split] = tokenized_datasets_nosent[split].remove_columns(["__index_level_0__"])

#  Sanity check
for split in tokenized_datasets_nosent.keys():
    print(f"{split} columns → {tokenized_datasets_nosent[split].column_names}")


### 7.2 : Training Setup for RoBERTa-Large

In [ ]:

# This cell defines the hyperparameters, metrics, and Trainer 
# for fine-tuning RoBERTa-Large on emotion classification WITHOUT sentiment features.

from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

#  Data collator dynamically pads sequences within each batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#  Training arguments
training_args = TrainingArguments(
    output_dir="roberta_large_nosent_results",
    eval_strategy="epoch",                
    save_strategy="epoch",                     
    learning_rate=1e-5,                         
    num_train_epochs=4,
    per_device_train_batch_size=32,              
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,                
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
    seed=42
)

#  Evaluation metrics
def compute_metrics(eval_pred):
    """Compute accuracy, macro F1, and weighted F1."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

#  Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets_nosent["train"],      
    eval_dataset=tokenized_datasets_nosent["validation"],  
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)



### 7.3 : Training ROberta Large 

In [ ]:
print(" Trainer ready — starting RoBERTa-Large.")
trainer.train()

### 7.4 : Evaluation on Test Set (RoBERTa-Large)

In [ ]:
# Evaluate RoBERTa-Large 
# Generates predictions, prints classification report, F1 scores,
# and plots a confusion matrix for detailed error analysis.

import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

#  Run predictions on the test split
predictions = trainer.predict(tokenized_datasets_nosent["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

#  Classification report
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

#  Store key metrics for summary table
roberta_large_acc = round(accuracy_score(y_true, y_pred), 3)
roberta_large_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
roberta_large_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print summary
print("\nSummary Metrics ")
print(f"Accuracy   : {roberta_large_acc:.3f}")
print(f"Macro F1   : {roberta_large_f1_macro:.3f}")
print(f"Weighted F1: {roberta_large_f1_weighted:.3f}")

#  Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred, labels=range(len(label_list)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_list, yticklabels=label_list)
plt.title("Confusion Matrix — RoBERTa-Large (No Sentiment)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

print(" Evaluation complete.")


-------------------------------------------

# 9 : RoBERTa-Large (Weighted Loss)


### 9.1 : RoBERTa-Large  and Weighted Cross-Entropy Loss for Class Imbalance Handling

In [ ]:
## RoBERTa-Large (Weighted Loss)
# This version adds class-weighted cross-entropy loss to handle label imbalance.
# Minority classes are penalized more heavily — no sentiment features.

import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModel

#  Label setup and class weights

label_list = sorted(train_df["Emotion"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}

# Convert training labels to integer IDs
y_train_int = y_train.map(label2id).to_numpy()

# Compute balanced class weights (clip extremes for stability)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label_list)),
    y=y_train_int,
)
weights = np.clip(weights, 0.7, 2.5) 

# Move weights to device
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print(" Class Weights:")
for lbl, w in zip(label_list, class_weights.tolist()):
    print(f"  {lbl:<10s}: {w:.4f}")

#  Define the weighted-loss RoBERTa-Large model

class RobertaWeighted(nn.Module):
    def __init__(self, checkpoint, num_labels, class_weights):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.roberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_labels),
        )

        self.class_weights = class_weights

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # [CLS] token
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

#  Initialize the weighted-loss model
checkpoint = "roberta-large"
model = RobertaWeighted(
    checkpoint=checkpoint,
    num_labels=len(label_list),
    class_weights=class_weights,
).to(device)

print("\n Weighted-loss RoBERTa-Large model initialized successfully.")


### 9.2 : Tokenization & Dataset Creation for RoBERTa-Large (Weighted)

In [ ]:

# Step 4 — Tokenization & Dataset Creation for RoBERTa-Large (Weighted)
# Goal: Convert text to token IDs and prepare HF DatasetDict


from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding

#  Load RoBERTa-Large tokenizer
checkpoint = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print(" Tokenizer loaded successfully!")

#  Tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["Sentence"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

#  Convert data into Hugging Face Datasets
train_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_train,
    "label": y_train.map(label2id)
}))
val_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_val,
    "label": y_val.map(label2id)
}))
test_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_test,
    "label": y_test.map(label2id)
}))

#  Combine into one DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

#  Apply tokenization
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Sentence"]
)

#  Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#  Remove unnecessary columns if they exist
for split in tokenized_datasets.keys():
    cols_to_drop = [c for c in ["token_type_ids", "__index_level_0__"] 
                    if c in tokenized_datasets[split].column_names]
    if cols_to_drop:
        tokenized_datasets[split] = tokenized_datasets[split].remove_columns(cols_to_drop)

#  Sanity check
for split in tokenized_datasets:
    print(f"{split} → {tokenized_datasets[split].column_names}")


### 9.3 : Training Setup and Evaluation Metrics for Weighted RoBERTa-Large with Sentiment Features

In [ ]:

#  Training Setup and Evaluation Metrics for Weighted RoBERTa-Large
#  Uses class-weighted cross-entropy loss (handled in model)
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# Step 1: Define training arguments
training_args = TrainingArguments(
    output_dir="roberta_large_weighted_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
    seed=42
)

# Step 2: Metric computation
def compute_metrics(eval_pred):
    """Compute accuracy and F1 metrics for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# Step 3: Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 4: Clean datasets (remove index columns safely)
if "tokenized_datasets" in locals():
    for split in tokenized_datasets.keys():
        cols_to_drop = [
            c for c in ["__index_level_0__", "text", "labels"]
            if c in tokenized_datasets[split].column_names
        ]
        if cols_to_drop:
            tokenized_datasets[split] = tokenized_datasets[split].remove_columns(cols_to_drop)

    print(" Cleaned dataset columns:")
    for split in tokenized_datasets.keys():
        print(f"  {split} → {tokenized_datasets[split].column_names}")
else:
    raise ValueError("tokenized_datasets not found. Please run the tokenization cell first.")

# Step 5: Initialize Trainer (no sampler — loss handles weights)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n Trainer for Weighted RoBERTa-Large initialized successfully.")


#### 9.4 : training the model 

In [ ]:
trainer.train()

### 9 .5 : Evaluate on Test Set

In [ ]:

#  Step 7.3 — Evaluate Weighted RoBERTa-Large Model

from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Step 1: Run predictions on the test split
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Step 2: Detailed classification report
print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

# Step 3: Store metrics for results table
roberta_large_weighted_acc = round(accuracy_score(y_true, y_pred), 3)
roberta_large_weighted_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
roberta_large_weighted_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print summary
print("\nTest Performance Summary ")
print(f"Accuracy:     {roberta_large_weighted_acc:.3f}")
print(f"Macro F1:     {roberta_large_weighted_f1_macro:.3f}")
print(f"Weighted F1:  {roberta_large_weighted_f1_weighted:.3f}")

# Step 4: Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_list,
    yticklabels=label_list
)
plt.title("Confusion Matrix — Weighted RoBERTa-Large")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

print("\nEvaluation complete.")


# 10 : DeBERTa-v3-Large (Weighted & Augmentation )

In [ ]:

# Step 1 — Environment Setup for DeBERTa-v3-Large (Weighted Training)
# Goal: Prepare libraries, device, and random seeds

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

#  Download VADER lexicon quietly
nltk.download("vader_lexicon", quiet=True)

#  Initialize sentiment analyzer (if later needed)
sia = SentimentIntensityAnalyzer()
#  Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")


### 10.1 Data Augmentation for Minority Classes

In [ ]:

#  Step 2 — Data Augmentation for Minority Classes
#  Goal: Expand underrepresented emotion categories using nlpaug

!pip install -q nlpaug

import nlpaug.augmenter.word as naw
import pandas as pd
import nltk

#  Ensure required NLTK resources are available
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

# Combine text + labels into a DataFrame
train_split_df = pd.DataFrame({"Sentence": X_train, "Emotion": y_train})

print("Before augmentation:")
print(train_split_df["Emotion"].value_counts())

# Define minority emotion classes (fewest samples)
minor_classes = ["disgust", "surprise", "anger"]

# Initialize synonym-based augmenter (WordNet-based)
aug = naw.SynonymAug(aug_src="wordnet", aug_p=0.3)

augmented_rows = []

# Augment each minority class until it reaches ~2000 samples
TARGET_SIZE = 10000
for emotion in minor_classes:
    subset = train_split_df[train_split_df["Emotion"] == emotion].copy()
    print(f"\n Augmenting '{emotion}' ({len(subset)} → {TARGET_SIZE})")

    while len(subset) < TARGET_SIZE:
        # Sample a fraction for augmentation
        new_sentences = subset["Sentence"].sample(frac=0.5, replace=True, random_state=42).tolist()

        # Apply augmentation
        new_aug = [aug.augment(s) for s in new_sentences]

        # Ensure each augmented sentence is a plain string (not a list)
        new_aug = [" ".join(s) if isinstance(s, list) else s for s in new_aug]

        # Add to subset
        temp_df = pd.DataFrame({"Sentence": new_aug, "Emotion": emotion})
        subset = pd.concat([subset, temp_df], ignore_index=True)

    augmented_rows.append(subset.head(TARGET_SIZE))

#  Merge augmented subsets with the rest of the training data
aug_df = pd.concat(augmented_rows)
train_balanced_df = (
    pd.concat([train_split_df, aug_df])
    .drop_duplicates(subset=["Sentence", "Emotion"])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("\nAfter augmentation:")
print(train_balanced_df["Emotion"].value_counts())
print(" Final balanced training size:", len(train_balanced_df))


### 10.2 : Weighted DeBERTa-v3-Large Model Definition

In [ ]:
# =====================================================
#  DeBERTa-v3-Large (Weighted Loss)
# =====================================================

import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModel

# -----------------------------------------------------
#  Label setup and class weights
# -----------------------------------------------------
label_list = sorted(train_df["Emotion"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Convert training labels to integer IDs
y_train_int = y_train.map(label2id).to_numpy()

# Compute balanced class weights (clip extremes for stability)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label_list)),
    y=y_train_int,
)
weights = np.clip(weights, 0.8, 2.5)  # optional stabilization

# Move weights to device
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print(" Class Weights:")
for lbl, w in zip(label_list, class_weights.tolist()):
    print(f"  {lbl:<10s}: {w:.4f}")

# -----------------------------------------------------
#  Define the weighted-loss DeBERTa-v3-Large model
# -----------------------------------------------------
class DebertaWeighted(nn.Module):
    def __init__(self, checkpoint, num_labels, class_weights):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.deberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_labels),
        )

        self.class_weights = class_weights

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # [CLS] token
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

# -----------------------------------------------------
#  Initialize weighted-loss DeBERTa model
# -----------------------------------------------------
checkpoint = "microsoft/deberta-v3-large"
model = DebertaWeighted(
    checkpoint=checkpoint,
    num_labels=len(label_list),
    class_weights=class_weights,
).to(device)

print("\n Weighted-loss DeBERTa-v3-Large model initialized successfully.")


### 10.3 Tokenization & Dataset Creation for DeBERTa-v3-Large

In [ ]:

#  Step 4 — Tokenization & Dataset Creation for DeBERTa-v3-Large
# Goal: Convert text to token IDs and prepare HF DatasetDict

!pip install -q tiktoken sentencepiece protobuf
!pip install -q sentencepiece

from datasets import Dataset, DatasetDict
from transformers import DebertaV2Tokenizer, DataCollatorWithPadding

#  Load DeBERTa-v3-Large tokenizer
checkpoint = "microsoft/deberta-v3-large"
tokenizer = DebertaV2Tokenizer.from_pretrained(checkpoint)
print(" Tokenizer loaded successfully!")

#  Tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["Sentence"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

#  Convert data into Hugging Face Datasets
train_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": train_balanced_df["Sentence"],
    "label": train_balanced_df["Emotion"].map(label2id)
}))
val_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_val,
    "label": y_val.map(label2id)
}))
test_dataset = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_test,
    "label": y_test.map(label2id)
}))

#  Combine into one DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

#  Apply tokenization
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Sentence"]
)

#  Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#  Remove token_type_ids and any leftover index columns
for split in tokenized_datasets.keys():
    remove_cols = []
    for col in ["token_type_ids", "__index_level_0__"]:
        if col in tokenized_datasets[split].column_names:
            remove_cols.append(col)
    if remove_cols:
        tokenized_datasets[split] = tokenized_datasets[split].remove_columns(remove_cols)

#  Sanity check
for split in tokenized_datasets:
    print(f"{split} columns → {tokenized_datasets[split].column_names}")


### 10. 4 : Training Configuration

In [ ]:
# Step 5 — Training Configuration + Evaluation for DeBERTa-v3-Large (Weighted)
# Goal: Train with class weights (no sampler)

from transformers import EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="deberta_v3_large_weighted_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=8e-6,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nTrainer for DeBERTa-v3-Large (Weighted) initialized successfully.")


### 10.5 : Training the Model

In [ ]:
#  Train the model
print(" Training started...")
trainer.train()
print("\n Training complete.")

### 10 .6  Evaluate on test set

In [ ]:
# Evaluate DeBERTa-v3-Large (Weighted)
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#  Run predictions on the test set
print("\nEvaluating on test set...")
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

#  Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

#  Store metrics for final results table
deberta_large_acc = round(accuracy_score(y_true, y_pred), 3)
deberta_large_f1_macro = round(f1_score(y_true, y_pred, average="macro"), 3)
deberta_large_f1_weighted = round(f1_score(y_true, y_pred, average="weighted"), 3)

# Print summary
print(f"\nTest Performance Summary:")
print(f"Accuracy:     {deberta_large_acc}")
print(f"Macro F1:     {deberta_large_f1_macro}")
print(f"Weighted F1:  {deberta_large_f1_weighted}")

#  Confusion matrix visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_list,
    yticklabels=label_list,
    cbar=False
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — DeBERTa-v3-Large (Weighted)")
plt.tight_layout()
plt.show()

print("\nEvaluation complete.")


# 11 : DeBERTa-v3-Large (Weighted, No Augmentation)

### 11.1 : Prepare labels + class weights

In [ ]:

# Step 1 — Prepare label mapping and compute class weights


import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# Unique label list and mapping
label_list = sorted(y_train.unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Convert y_train to integer IDs
y_train_int = y_train.map(label2id).to_numpy()

# Compute class weights (clip for stability)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label_list)),
    y=y_train_int,
)
weights = np.clip(weights, 0.8, 2.5)

# Send weights to device
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print(" Class Weights:")
for lbl, w in zip(label_list, class_weights.tolist()):
    print(f"  {lbl:<10s}: {w:.4f}")


### 11.2 : Define the DeBERTa-v3-Large weighted model

In [ ]:

# Step 2 — Define the DeBERTa-v3-Large model with weighted loss


import torch.nn as nn
from transformers import AutoModel

class DebertaWeighted(nn.Module):
    def __init__(self, checkpoint, num_labels, class_weights):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.deberta.config.hidden_size

        # Dropout layers for regularization
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, num_labels),
        )

        self.class_weights = class_weights

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

# Initialize model
checkpoint = "microsoft/deberta-v3-large"
model = DebertaWeighted(checkpoint, num_labels=len(label_list), class_weights=class_weights).to(device)

print("\n Weighted DeBERTa-v3-Large model initialized successfully.")


### 11.3 : Tokenization & Dataset Creation (No Augmentation)

In [ ]:

# Step 3 — Tokenization & Dataset Creation (DeBERTa-v3-Large Weighted No Augmentation)


!pip install -q sentencepiece protobuf

from datasets import Dataset, DatasetDict
from transformers import DebertaV2Tokenizer, DataCollatorWithPadding

# Load tokenizer
checkpoint = "microsoft/deberta-v3-large"
tokenizer = DebertaV2Tokenizer.from_pretrained(checkpoint)
print(" Tokenizer loaded successfully!")

# Tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["Sentence"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

# Convert splits into Hugging Face Datasets
train_dataset_dbert_v3_weighted_noaug = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_train,
    "label": y_train.map(label2id)
}))
val_dataset_dbert_v3_weighted_noaug = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_val,
    "label": y_val.map(label2id)
}))
test_dataset_dbert_v3_weighted_noaug = Dataset.from_pandas(pd.DataFrame({
    "Sentence": X_test,
    "label": y_test.map(label2id)
}))

# Combine into DatasetDict
dataset_dbert_v3_weighted_noaug = DatasetDict({
    "train": train_dataset_dbert_v3_weighted_noaug,
    "validation": val_dataset_dbert_v3_weighted_noaug,
    "test": test_dataset_dbert_v3_weighted_noaug,
})

# Apply tokenization
tokenized_datasets_dbert_v3_weighted_noaug = dataset_dbert_v3_weighted_noaug.map(
    tokenize_function,
    batched=True,
    remove_columns=["Sentence"]
)

# Dynamic padding
data_collator_dbert_v3_weighted_noaug = DataCollatorWithPadding(tokenizer=tokenizer)

# Clean up unneeded columns if any
for split in tokenized_datasets_dbert_v3_weighted_noaug.keys():
    cols_to_drop = [c for c in ["token_type_ids", "__index_level_0__"] 
                    if c in tokenized_datasets_dbert_v3_weighted_noaug[split].column_names]
    if cols_to_drop:
        tokenized_datasets_dbert_v3_weighted_noaug[split] = tokenized_datasets_dbert_v3_weighted_noaug[split].remove_columns(cols_to_drop)

# Sanity check
for split in tokenized_datasets_dbert_v3_weighted_noaug:
    print(f"{split} → {tokenized_datasets_dbert_v3_weighted_noaug[split].column_names}")


### 11.4 : Training Configuration & Trainer Setup (Weighted, No Augmentation)

In [ ]:

# Step 4 — Training Configuration & Trainer Setup (DeBERTa-v3-Large Weighted No Augmentation)


from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Training arguments
training_args_dbert_v3_weighted_noaug = TrainingArguments(
    output_dir="deberta_v3_large_weighted_noaug_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=8e-6,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    dataloader_pin_memory=False,
)

# Metric computation
def compute_metrics_dbert_v3_weighted_noaug(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# Initialize Trainer
trainer_dbert_v3_weighted_noaug = Trainer(
    model=model,
    args=training_args_dbert_v3_weighted_noaug,
    train_dataset=tokenized_datasets_dbert_v3_weighted_noaug["train"],
    eval_dataset=tokenized_datasets_dbert_v3_weighted_noaug["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator_dbert_v3_weighted_noaug,
    compute_metrics=compute_metrics_dbert_v3_weighted_noaug,
)

print("\n Trainer for DeBERTa-v3-Large (Weighted No Augmentation) initialized successfully.")


### 11 .5 : Train the Model (DeBERTa-v3-Large Weighted No Augmentation)

In [ ]:

# Step 5 — Train the Model (DeBERTa-v3-Large Weighted No Augmentation)


print(" Training started for DeBERTa-v3-Large (Weighted No Augmentation)...")
train_result_dbert_v3_weighted_noaug = trainer_dbert_v3_weighted_noaug.train()
print("\n Training complete!")


### 11.6 : Evaluate on Test Set (DeBERTa-v3-Large Weighted No Augmentation)

In [ ]:

# Step 6 — Evaluate on Test Set (DeBERTa-v3-Large Weighted No Augmentation)


from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Evaluate on test split
print(" Evaluating on test data...")
test_results_dbert_v3_weighted_noaug = trainer_dbert_v3_weighted_noaug.evaluate(
    tokenized_datasets_dbert_v3_weighted_noaug["test"]
)

print("\n Test Evaluation Metrics:")
print(test_results_dbert_v3_weighted_noaug)

# Predictions for detailed metrics
predictions_dbert_v3_weighted_noaug = trainer_dbert_v3_weighted_noaug.predict(
    tokenized_datasets_dbert_v3_weighted_noaug["test"]
)
preds = np.argmax(predictions_dbert_v3_weighted_noaug.predictions, axis=-1)
labels = predictions_dbert_v3_weighted_noaug.label_ids

# Classification report
report = classification_report(labels, preds, target_names=label_list)
print("\n=== Classification Report: DeBERTa-v3-Large (Weighted No Augmentation) ===\n")
print(report)

# Confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_list, yticklabels=label_list)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — DeBERTa-v3-Large (Weighted No Augmentation)')
plt.show()

# Summary metrics
acc = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")
f1_weighted = f1_score(labels, preds, average="weighted")

print("\nTest Performance Summary:")
print(f"Accuracy:     {acc:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")
print(f"Weighted F1:  {f1_weighted:.3f}")

# Store metrics for the final summary table
deberta_v3_weighted_noaug_acc = acc
deberta_v3_weighted_noaug_f1_macro = f1_macro
deberta_v3_weighted_noaug_f1_weighted = f1_weighted


| **Model**                                       | **Accuracy** | **Macro F1** | **Weighted F1** | **Notes**                                |
| ----------------------------------------------- | ------------ | ------------ | --------------- | ---------------------------------------- |
| **BERT-base**                                   | 0.588        | 0.483        | 0.580           | Baseline model                           |
| **RoBERTa-base**                                | 0.616        | 0.561        | 0.609           | Stronger contextual encoding             |
| **RoBERTa-base + Sentiments**                   | 0.613        | 0.588        | 0.605           | Added VADER sentiment features           |
| **RoBERTa-Large**                               | 0.672        | 0.596        | 0.668           | Larger model, better context             |
| **Weighted RoBERTa-Large**                      | 0.693        | 0.632        | 0.692           | Class-balanced loss improves rare labels |
| **DeBERTa-v3-Large (Weighted + Augmentation)**  | 0.652        | 0.554        | 0.647           | Data augmentation on minorities          |
| **DeBERTa-v3-Large (Weighted No Augmentation)** | 0.654        | 0.549        | 0.651           | No augmentation baseline                 |
